In [9]:
pip install scikit-surprise

In [7]:
import pandas as pd

from surprise import Dataset, Reader
from surprise import SVD, SVDpp, NMF
from surprise.model_selection import GridSearchCV, cross_validate

ratings = pd.read_csv("ratings.csv")

ratings.head()
print(ratings.head())

ratings.info()
print(ratings.info())

ratings["rating"].describe()
print(ratings["rating"].describe())

reader = Reader(rating_scale=(0.5, 5.0))

data = Dataset.load_from_df(
    ratings[["userId", "movieId", "rating"]],
    reader
)

param_grid_svd = {
    "n_factors": [50, 100],
    "n_epochs": [20],
    "lr_all": [0.005],
    "reg_all": [0.02, 0.05]
}

grid_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=["rmse", "mae"],
    cv=2,
    n_jobs=-1
)

grid_svd.fit(data)

print("Best SVD RMSE:", grid_svd.best_score["rmse"])
print("Best SVD params:", grid_svd.best_params["rmse"])

param_grid_svdpp = {
    "n_factors": [20],
    "n_epochs": [10],
    "lr_all": [0.005],
    "reg_all": [0.02]
}

grid_svdpp = GridSearchCV(
    SVDpp,
    param_grid_svdpp,
    measures=["rmse", "mae"],
    cv=2,
    n_jobs=-1
)

grid_svdpp.fit(data)

print("Best SVD++ RMSE:", grid_svdpp.best_score["rmse"])
print("Best SVD++ params:", grid_svdpp.best_params["rmse"])

param_grid_nmf = {
    "n_factors": [15, 30],
    "n_epochs": [20],
    "reg_pu": [0.02],
    "reg_qi": [0.02]
}

grid_nmf = GridSearchCV(
    NMF,
    param_grid_nmf,
    measures=["rmse", "mae"],
    cv=2,
    n_jobs=-1
)

grid_nmf.fit(data)

print("Best NMF RMSE:", grid_nmf.best_score["rmse"])
print("Best NMF params:", grid_nmf.best_params["rmse"])

results = pd.DataFrame({
    "Algorithm": ["SVD", "SVD++", "NMF"],
    "Best RMSE": [
        grid_svd.best_score["rmse"],
        grid_svdpp.best_score["rmse"],
        grid_nmf.best_score["rmse"]
    ],
    "Best MAE": [
        grid_svd.best_score["mae"],
        grid_svdpp.best_score["mae"],
        grid_nmf.best_score["mae"]
    ],
    "Best params": [
        grid_svd.best_params["rmse"],
        grid_svdpp.best_params["rmse"],
        grid_nmf.best_params["rmse"]
    ]
})

results.sort_values("Best RMSE")

best_model = SVD(**grid_svd.best_params["rmse"])

trainset = data.build_full_trainset()
best_model.fit(trainset)

user_id = 1
movie_id = 1

prediction = best_model.predict(user_id, movie_id)

prediction

print(f"Прогнозований рейтинг: {prediction.est:.2f}")

   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: fl

In [ ]:
'''
У роботі було використано датасет MovieLens, який містить оцінки користувачів для фільмів.
Для побудови рекомендаційної системи були протестовані алгоритми матричної факторизації SVD, SVD++ та NMF.

Для кожного алгоритму було виконано підбір гіперпараметрів за допомогою GridSearchCV з крос-валідацією.
Якість моделей порівнювалася за метриками RMSE та MAE.

Оптимальною моделлю є та, яка має найменше значення RMSE, оскільки ця метрика показує середню помилку прогнозування рейтингу.
Зазвичай SVD або SVD++ показують кращі результати за NMF, але SVD++ працює повільніше.
Якщо різниця між SVD та SVD++ невелика, доцільно обрати SVD, бо він швидший і простіший у використанні.
'''